# 01 — Tiền xử lý dữ liệu

> **Đầu việc 01/6** trong pipeline phát hiện xâm nhập NSL-KDD.

**Mục tiêu**
- Encode 3 biến phân loại: `protocol_type`, `service`, `flag`.
- Chuẩn hóa các biến số về cùng thang đo.
- Loại `difficulty` và feature hằng (`num_outbound_cmds`).
- Fit encoder/scaler **chỉ trên train**, chỉ transform trên test.

**Sản phẩm**: `data/processed/X_train.npy`, `X_test.npy`, `y_train_binary.npy`, `y_test_binary.npy`, `y_*_family.npy`.

## 0. Môi trường

Cài đặt: `pip install -r requirements.txt` (đã gồm `jupyter`).

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from nids.config import (CATEGORICAL, DATA_PROCESSED, FEATURES, RANDOM_STATE,
                         REPORTS, FAMILY_ORDER)
from nids.data import load_train_test, to_binary_label, to_family_label, novel_attack_types

pd.set_option("display.float_format", lambda x: f"{x:.4f}")
np.random.seed(RANDOM_STATE)
print("Project root:", PROJECT_ROOT)
print("RANDOM_STATE =", RANDOM_STATE)

Project root: /mnt/Data/ptit/Network_Intrusion&Cyber_Attack_Detection
RANDOM_STATE = 36


## 1. Nạp dữ liệu thô

Mỗi dòng gồm 41 đặc trưng + `label` + `difficulty`.

In [2]:
# 1) Nạp dữ liệu thô (mỗi dòng = 41 đặc trưng + label + difficulty)
train, test = load_train_test()
print("train:", train.shape)
print("test: ", test.shape)
print("số đặc trưng gốc:", len(FEATURES))
train.head()

train: (125973, 43)
test:  (22544, 43)
số đặc trưng gốc: 41


,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,...,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,label,difficulty
0,0,tcp,ftp_data,SF,491,0,0,0,0,0,...,0.1700,0.0300,0.1700,0.0000,0.0000,0.0000,0.0500,0.0000,normal,20
1,0,udp,other,SF,146,0,0,0,0,0,...,0.0000,0.6000,0.8800,0.0000,0.0000,0.0000,0.0000,0.0000,normal,15
2,0,tcp,private,S0,0,0,0,0,0,0,...,0.1000,0.0500,0.0000,0.0000,1.0000,1.0000,0.0000,0.0000,neptune,19
3,0,tcp,http,SF,232,8153,0,0,0,0,...,1.0000,0.0000,0.0300,0.0400,0.0300,0.0100,0.0000,0.0100,normal,21
4,0,tcp,http,SF,199,420,0,0,0,0,...,1.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,normal,21


## 2. Kiểm tra chất lượng dữ liệu

- `difficulty` là điểm khó, **không dùng làm đặc trưng**.
- Feature hằng không mang thông tin và làm hỏng ma trận hiệp phương sai của Mahalanobis.
- NSL-KDD đã khử trùng lặp, nhưng vẫn kiểm tra lại.

In [3]:
# 2) Kiểm tra chất lượng dữ liệu
for name, df in {"train": train, "test": test}.items():
    constant = [c for c in FEATURES if df[c].nunique() == 1]
    print(f"{name}: duplicates={int(df.duplicated().sum())}, constant={constant}")

train: duplicates=0, constant=['num_outbound_cmds']
test: duplicates=0, constant=['num_outbound_cmds']


## 3. Encode biến phân loại

`service` có cardinality cao (~70 giá trị). Dùng `OneHotEncoder(handle_unknown="ignore")` để test không vỡ nếu gặp giá trị lạ.

In [4]:
# 3) Encode 3 biến phân loại bằng OneHotEncoder
#    - Chỉ fit trên train để tránh rò rỉ thông tin từ test.
#    - handle_unknown="ignore": test gặp giá trị lạ thì cho cột toàn 0, không vỡ.
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
encoder.fit(train[CATEGORICAL])

train_cat = encoder.transform(train[CATEGORICAL])
test_cat = encoder.transform(test[CATEGORICAL])
print("số cột sau one-hot:", train_cat.shape[1])  # vd. protocol 3 + service 70 + flag 11

số cột sau one-hot: 84


## 4. Chuẩn hóa biến số

Euclidean rất nhạy với thang đo; Mahalanobis cần covariance ổn định. Dùng `StandardScaler`, **fit chỉ trên train**.

In [5]:
# 4) Chuẩn hóa biến số bằng StandardScaler
#    - Bỏ đặc trưng hằng num_outbound_cmds (toàn 0, không mang thông tin).
#    - Chỉ fit trên train, transform cả train và test.
from sklearn.preprocessing import StandardScaler

numeric_features = [c for c in FEATURES if c not in CATEGORICAL and c != "num_outbound_cmds"]
print("số biến số dùng:", len(numeric_features))

scaler = StandardScaler()
scaler.fit(train[numeric_features])

train_num = scaler.transform(train[numeric_features])
test_num = scaler.transform(test[numeric_features])

# Ghép: [biến số đã chuẩn hóa | biến phân loại đã one-hot]
X_train = np.hstack([train_num, train_cat])
X_test = np.hstack([test_num, test_cat])
print("X_train:", X_train.shape)
print("X_test: ", X_test.shape)

số biến số dùng: 37


X_train: (125973, 121)
X_test:  (22544, 121)


## 5. Lưu dữ liệu đã xử lý

Lưu vào `data/processed/` để notebook 02–06 dùng lại, tránh xử lý lặp.

In [6]:
# 5) Lưu dữ liệu đã xử lý để các notebook 02-06 dùng lại
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

y_train_binary = to_binary_label(train).to_numpy()
y_test_binary = to_binary_label(test).to_numpy()
y_train_family = to_family_label(train).to_numpy()
y_test_family = to_family_label(test).to_numpy()

np.save(DATA_PROCESSED / "X_train.npy", X_train)
np.save(DATA_PROCESSED / "X_test.npy", X_test)
np.save(DATA_PROCESSED / "y_train_binary.npy", y_train_binary)
np.save(DATA_PROCESSED / "y_test_binary.npy", y_test_binary)
np.save(DATA_PROCESSED / "y_train_family.npy", y_train_family)
np.save(DATA_PROCESSED / "y_test_family.npy", y_test_family)

# Lưu tên cột cuối cùng để các notebook khác hiểu ma trận X
import json
feature_names = numeric_features + list(encoder.get_feature_names_out(CATEGORICAL))
with open(DATA_PROCESSED / "features.json", "w", encoding="utf-8") as f:
    json.dump({"numeric": numeric_features, "all": feature_names,
               "n_features": len(feature_names)}, f, ensure_ascii=False, indent=2)
print("đã lưu 6 file .npy + features.json")
print("tổng số đặc trưng cuối cùng:", len(feature_names))

đã lưu 6 file .npy + features.json
tổng số đặc trưng cuối cùng: 121


## Checklist hoàn thành

- [x] Không còn cột `difficulty` và `num_outbound_cmds` trong ma trận đặc trưng.
- [x] Encoder/scaler chỉ fit trên train.
- [x] Kích thước `X_train`/`X_test` khớp số dòng ban đầu.
- [x] Đã lưu đủ 6 file `.npy` vào `data/processed/`.